# PyOccam Demo Notebook
## Clean Implementation for Search, Fit, and Analysis

This notebook demonstrates the core OCCAM workflow:
1. Load data (built-in or custom)
2. Search for best models
3. Fit selected models
4. Analyze results
5. Save outputs

In [25]:
# Import required libraries
import pyoccam
import os
import pandas as pd
import numpy as np
from datetime import datetime

print(f"PyOccam version: {pyoccam.__version__}")
print("\nAvailable built-in datasets:")
print("  • pyoccam.load_dementia()   - Alzheimer's/dementia risk factors")
print("  • pyoccam.load_landslides() - Geological hazard prediction")
print("\nQuick methods:")
print("  • pyoccam.quick_search()    - One-line search")
print("  • pyoccam.help()            - Show help")

PyOccam version: 0.1.2

Available built-in datasets:
  • pyoccam.load_dementia()   - Alzheimer's/dementia risk factors
  • pyoccam.load_landslides() - Geological hazard prediction

Quick methods:
  • pyoccam.quick_search()    - One-line search
  • pyoccam.help()            - Show help


## 1. Configuration

Set your analysis parameters here:

In [26]:
# Configuration parameters
DATA_SOURCE = "landslides"                # Options: "dementia", "landslides", or path to your file
SEARCH_TYPE = "full-up"            # Options: loopless-up, full-up, disjoint-up, chain-up
SEARCH_LEVELS = 7                      # Search depth (1-10+)
SEARCH_WIDTH = 3                       # Beam width (1-20+)
OUTPUT_DIR = "occam_output"            # Directory for output files

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Configuration:")
print(f"  Data source: {DATA_SOURCE}")
print(f"  Search type: {SEARCH_TYPE}")
print(f"  Levels: {SEARCH_LEVELS}, Width: {SEARCH_WIDTH}")
print(f"  Output directory: {OUTPUT_DIR}")

Configuration:
  Data source: landslides
  Search type: full-up
  Levels: 7, Width: 3
  Output directory: occam_output


## 2. Load Data

PyOccam provides built-in datasets and also supports custom data files:

In [27]:
# Load data using built-in datasets or custom file
if DATA_SOURCE.lower() == "dementia":
    # Use built-in dementia dataset
    data = pyoccam.load_dementia()
    print("Using built-in dementia dataset")
elif DATA_SOURCE.lower() == "landslides":
    # Use built-in landslides dataset
    data = pyoccam.load_landslides()
    print("Using built-in landslides dataset")
else:
    # Load custom data file
    data = pyoccam.load_data(DATA_SOURCE)
    print(f"Loaded custom data file: {DATA_SOURCE}")

# Get the manager from the data object
manager = data.manager

# Display data information
print(f"\n✓ Dataset: {data.DESCR if hasattr(data, 'DESCR') else DATA_SOURCE}")
print(f"  Sample size: {data.n_samples}")
print(f"  Features: {data.n_features}")
print(f"  Target variable: {data.target_name}")
print(f"  Has test data: {data.has_test_data}")

# Show feature names
if hasattr(data, 'feature_names') and data.feature_names:
    print(f"\nFeature variables ({len(data.feature_names)}):")
    for i, var in enumerate(data.feature_names[:10], 1):  # Show first 10
        print(f"  {i:2d}. {var}")
    if len(data.feature_names) > 10:
        print(f"  ... and {len(data.feature_names) - 10} more")

✓ Loaded landslides: 1077 samples, 20 features
Using built-in landslides dataset

✓ Dataset: Landslides/Geological hazard dataset
  Sample size: 1077
  Features: 20
  Target variable: LS
  Has test data: True

Feature variables (20):
   1. ASpect_reclass
   2. CLay_reclass
   3. CurVature_reclass
   4. ELevation_reclass
   5. Fault_Dns_reclass
   6. HaBitatMap_UTM
   7. Geol_Lith
   8. nlcd_2021_Land_Cover
   9. Geol_rock_Type
  10. Rd_strm_Dns_reclass
  ... and 10 more


## Alternative: Different Ways to Load Data

PyOccam provides multiple ways to load data:

In [14]:
# Method 1: Built-in datasets (recommended for getting started)
# dementia_data = pyoccam.load_dementia()
# landslides_data = pyoccam.load_landslides()

# Method 2: Load custom data file
# custom_data = pyoccam.load_data("mydata.txt")

# Method 3: Traditional VBMManager approach (for advanced users)
# manager = pyoccam.VBMManager()
# manager.init_from_command_line(["occam", "datafile.txt"])

# All methods provide access to the manager:
# manager = data.manager  # From data object

print("Data loading methods demonstrated (commented out)")

Data loading methods demonstrated (commented out)


## 3. Configure Report Variables

Set which statistics to include in reports:

In [15]:
# Configure report columns
# Note: Don't include 'ID' or 'Model' - they're added automatically
report_vars = "Level$I, h, ddf, dLR, Alpha, %C, %dH(DV), dAIC, dBIC"
manager.set_report_variables(report_vars)

print("Report configured with variables:")
print(f"  {report_vars}")
print("\nVariable meanings:")
print("  Level  = Search level")
print("  h      = Entropy")
print("  ddf    = Delta degrees of freedom (complexity)")
print("  dLR    = Delta log likelihood ratio")
print("  Alpha  = Statistical significance")
print("  %C     = Percent correct (classification accuracy)")
print("  %dH(DV)= Percent uncertainty reduction")
print("  dAIC   = Delta AIC (vs independence)")
print("  dBIC   = Delta BIC (vs independence)")

Report configured with variables:
  Level$I, h, ddf, dLR, Alpha, %C, %dH(DV), dAIC, dBIC

Variable meanings:
  Level  = Search level
  h      = Entropy
  ddf    = Delta degrees of freedom (complexity)
  dLR    = Delta log likelihood ratio
  Alpha  = Statistical significance
  %C     = Percent correct (classification accuracy)
  %dH(DV)= Percent uncertainty reduction
  dAIC   = Delta AIC (vs independence)
  dBIC   = Delta BIC (vs independence)


## 4. Run Search

Search for the best models using your chosen algorithm:

In [16]:
# Run search
print(f"Running {SEARCH_TYPE} search...")
print(f"  Levels: {SEARCH_LEVELS}")
print(f"  Width: {SEARCH_WIDTH}")
print("-" * 60)

search_report = manager.generate_search_report(
    search_type=SEARCH_TYPE,
    levels=SEARCH_LEVELS,
    width=SEARCH_WIDTH,
    include_test_data=False
)

# Display search results (first 50 lines)
lines = search_report.split('\n')
for line in lines[:50]:
    print(line)
if len(lines) > 50:
    print(f"\n... ({len(lines) - 50} more lines)")

Running full-up search...
  Levels: 7
  Width: 3
------------------------------------------------------------
Searching levels:
1 : 20 new models, 3 kept; 4 total kept
2 : 54 new models, 3 kept; 7 total kept
3 : 55 new models, 3 kept; 10 total kept
4 : 57 new models, 3 kept; 13 total kept
5 : 63 new models, 3 kept; 16 total kept
6 : 72 new models, 3 kept; 19 total kept
7 : 84 new models, 3 kept; 22 total kept

  ID   MODEL                                    Level              H            ddf            dLR          Alpha        %dH(DV)           daic           dbic      Inc.Alpha       %C(Data)         %cover       %C(Test)          %miss
  22*  IV:ElTwZ:HbZ:LcZ:SlZ:TcZ                     7        10.3926             12       508.9012         0.0000        38.0436       484.9012       425.1180         0.0009        77.9016        51.8519        72.7612         2.2388
  21*  IV:CvZ:ElZ:HbZ:GlZ:LcZ:SlZ:TcZ               7        10.3968             11       502.6400         0.0000     

## 5. Identify Best Model

Extract the best model based on BIC:

In [21]:
# Get best model
best_model = manager.get_best_model_by_bic()

if best_model:
    print(f"✨ Best model (by BIC): {best_model}")
    
    # Find its statistics in the search report
    for line in search_report.split('\n'):
        if best_model in line:
            print(f"\nModel statistics:")
            print(line)
            
            # Parse key metrics
            parts = line.split()
            if len(parts) > 10:
                try:
                    print(f"\nKey metrics for {best_model}:")
                    print(f"  • Percent Correct: {float(parts[7]):.2f}%")
                    print(f"  • Information Captured: {float(parts[8]):.2f}%")
                    print(f"  • BIC Improvement: {float(parts[10]):.4f}")
                    print(f"  • Significance (α): {float(parts[6]):.4f}")
                except:
                    pass
            break
else:
    print("No best model found")
    best_model = "IV:CaseControl"  # Fallback model

✨ Best model (by BIC): IV:CvZ:ElZ:HbZ:GlZ:LcZ:SlZ:TcZ

Model statistics:
  21*  IV:CvZ:ElZ:HbZ:GlZ:LcZ:SlZ:TcZ               7        10.3968             11       502.6400         0.0000        37.5755       480.6400       425.8387         0.0001        76.6017        36.2654        70.5224         6.7164

Key metrics for IV:CvZ:ElZ:HbZ:GlZ:LcZ:SlZ:TcZ:
  • Percent Correct: 37.58%
  • Information Captured: 480.64%
  • BIC Improvement: 0.0001
  • Significance (α): 0.0000


## 6. Fit Best Model

Generate detailed fit report for the best model:

In [22]:
# Fit the best model
print(f"Fitting model: {best_model}")
print("-" * 60)

fit_report = manager.generate_fit_report(
        model_name=best_model,
        target_state="0"
    )

# Display key sections of fit report
fit_lines = fit_report.split('\n')
show_lines = False
line_count = 0
max_lines = 100

for line in fit_lines:
    # Show important sections
    if any(key in line for key in ["Model", "Percent", "Entropy", "Information", "Alpha"]):
        show_lines = True
    
    if show_lines and line_count < max_lines:
        print(line)
        line_count += 1
        
        # Stop after contingency table
        if "Degrees of" in line:
            show_lines = False

print(f"\n... (Full report has {len(fit_lines)} lines)")

Fitting model: IV:CvZ:ElZ:HbZ:GlZ:LcZ:SlZ:TcZ
------------------------------------------------------------
    Model,IV:CvZ:ElZ:HbZ:GlZ:LcZ:SlZ:TcZ (Directed System)
    IV Component:,ASpect_reclass; CLay_reclass; CurVature_reclass; ELevation_reclass; Fault_Dns_reclass; HaBitatMap_UTM; Geol_Lith; nlcd_2021_Land_Cover; Geol_rock_Type; Rd_strm_Dns_reclass; SLope_reclass; TWi_reclass; GeomDesc; TaxOrder; TaxSuborder; taxGrtGroup; TaxSubgrp; taxPartSize; TaxClname; DrainageCl,AsClCvElFdHbGlLcGtRdSlTwGdToTsGgTs1PsTcDc
    Model Component: ,CurVature_reclass; LS,CvZ
    Model Component: ,ELevation_reclass; LS,ElZ
    Model Component: ,HaBitatMap_UTM; LS,HbZ
    Model Component: ,Geol_Lith; LS,GlZ
    Model Component: ,nlcd_2021_Land_Cover; LS,LcZ
    Model Component: ,SLope_reclass; LS,SlZ
    Model Component: ,TaxClname; LS,TcZ
    Degrees of Freedom (DF):,-1
    Entropy(H):,10.3968
    Information captured (%):,37.5755
    Transmission (T):,0.559289

---------------------------------------

## 7. Fit Custom Model (Optional)

You can also fit your own model specification:

In [ ]:
# Example: Fit a custom model
# Format: "IV:Var1Var2:Var3Var4" where variables form interaction terms

# Examples for built-in datasets:
# For dementia:   custom_model = "IV:ApSxZ:EdZ:CZ"  
# For landslides: custom_model = "IV:slZ:twZ:fdZ"
# Generic:        custom_model = "IV:ABC:DEF"

# To fit a custom model, uncomment below:
"""
custom_model = "IV:YourModelHere"
print(f"Fitting custom model: {custom_model}")

try:
    custom_report = manager.generate_fit_report(
        custom_model,
        use_ipf_start=False,
        skip_trained_model_table=False,
        skip_ivi_tables=False
    )
    
    # Show summary
    for line in custom_report.split('\n')[:30]:
        print(line)
        
except Exception as e:
    print(f"Error fitting model: {e}")
"""

print("To fit a custom model, edit the cell above and uncomment the code.")
print("\nVariable abbreviations for built-in datasets:")
if DATA_SOURCE == "dementia":
    print("  Dementia: Ap=APOE, Ed=Education, Ag=AgeLastExam, Z=CaseControl, etc.")
elif DATA_SOURCE == "landslides":
    print("  Landslides: sl=Slope, tw=TWI, fd=FaultDensity, cv=Curvature, el=Elevation")

## 8. Save Results to Files

Save search and fit reports for later analysis:

In [23]:
# Generate timestamp for unique filenames
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Determine dataset name for filename
dataset_name = DATA_SOURCE.replace('.txt', '').replace('/', '_')

# Save search report
search_file = os.path.join(OUTPUT_DIR, f"search_{dataset_name}_{SEARCH_TYPE}_{timestamp}.txt")
with open(search_file, 'w') as f:
    f.write(search_report)
print(f"✓ Search report saved to: {search_file}")

# Save fit report
fit_file = os.path.join(OUTPUT_DIR, f"fit_{dataset_name}_{best_model.replace(':', '_')}_{timestamp}.txt")
with open(fit_file, 'w') as f:
    f.write(fit_report)
print(f"✓ Fit report saved to: {fit_file}")

print(f"\n📁 All results saved in: {os.path.abspath(OUTPUT_DIR)}")

✓ Search report saved to: occam_output\search_landslides_full-up_20251004_004201.txt
✓ Fit report saved to: occam_output\fit_landslides_IV_CvZ_ElZ_HbZ_GlZ_LcZ_SlZ_TcZ_20251004_004201.txt

📁 All results saved in: D:\projects\occam\pyoccam\occam_output


## 9. Parse Results for Analysis

Extract search results into a pandas DataFrame for further analysis:

In [24]:
# Parse search results into DataFrame
def parse_search_results(report):
    """Parse OCCAM search report into DataFrame"""
    lines = report.split('\n')
    data_lines = []
    
    # Find data section
    in_data = False
    for line in lines:
        if 'Model' in line and 'Level' in line:
            in_data = True
            continue
        if in_data and line.strip() and not line.startswith('#'):
            parts = line.split()
            if len(parts) >= 8:  # Valid data line
                data_lines.append(parts)
    
    if data_lines:
        # Create DataFrame
        columns = ['ID', 'Model', 'Level', 'h', 'ddf', 'dLR', 'Alpha', '%C', '%dH(DV)', 'dAIC', 'dBIC']
        df = pd.DataFrame(data_lines, columns=columns[:len(data_lines[0])])
        
        # Convert numeric columns
        for col in ['Level', 'ddf', 'dLR', 'Alpha', '%C', '%dH(DV)', 'dAIC', 'dBIC']:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')
        
        return df
    return None

# Parse and display results
df = parse_search_results(search_report)
if df is not None:
    print(f"Parsed {len(df)} models from search results\n")
    print("Top 10 models by dBIC:")
    display_cols = ['Model', 'Level', 'dBIC', '%C', 'Alpha']
    print(df.nlargest(10, 'dBIC')[display_cols])
else:
    print("Could not parse search results")

Could not parse search results


## 10. Summary

Analysis complete! You have:
- Used built-in datasets or loaded custom data
- Searched for best models using OCCAM
- Identified the best model by BIC
- Generated detailed fit reports
- Saved results to files
- Parsed results for further analysis

### Next Steps:
1. Try different built-in datasets (dementia, landslides)
2. Experiment with search algorithms (full-up, disjoint-up)
3. Adjust search parameters (levels, width)
4. Fit custom models based on domain knowledge
5. Visualize results (see visualization notebook)
6. Compare models across different datasets

### Quick Reference:
```python
# Built-in datasets
data = pyoccam.load_dementia()    # Alzheimer's risk factors
data = pyoccam.load_landslides()  # Geological hazards

# Quick search
data, best = pyoccam.quick_search("dementia", "loopless-up", 7, 3)

# Access manager from data
manager = data.manager

# Data object attributes
print(data.n_samples)      # Number of samples
print(data.n_features)     # Number of features
print(data.feature_names)  # Variable names
print(data.target_name)    # Dependent variable
```

## Tips and Tricks

### Working with Built-in Data:
- Built-in datasets come with meaningful variable names and descriptions
- Use `data.DESCR` to see dataset description
- Use `data.feature_names` to see all variable names

### Understanding Model Notation:
- `IV` = Independence model (all variables independent)
- `:` separates relations (groups of interacting variables)
- Variables within a relation interact (e.g., `ABC` means A, B, C interact)
- Example: `IV:ApZ:EdZ` means APOE and CaseControl interact, Education and CaseControl interact

### Search Strategies:
- Start with `loopless-up` for faster initial exploration
- Use `full-up` for comprehensive search
- Increase levels/width for more thorough search (but slower)
- Higher dBIC = better model (positive is good, negative is bad)

### Performance Tips:
- Start with small searches (levels=3-5, width=3) for quick results
- Save intermediate results to avoid re-running long searches
- Use `data.quick_search()` for rapid prototyping

### Interpreting Results:
- **%C > 70%**: Good classification accuracy
- **%C > 80%**: Very good accuracy
- **%C > 90%**: Excellent accuracy
- **dBIC > 10**: Strong evidence for model
- **Alpha < 0.05**: Statistically significant
- **%dH(DV) > 20%**: Substantial information captured

Remember: Balance accuracy (%C) with model complexity (ddf) and interpretability!